# Fine-Tuning Final IndoBERT ABSA - c4_p1_cw_ls Only

Notebook ini adalah versi final/penyelamatan untuk skripsi. Notebook hanya menjalankan satu konfigurasi model final `c4_p1_cw_ls`, yaitu konfigurasi terbaik dari eksperimen `finetune2-ex2`, agar tidak perlu mengulang 10 eksperimen penuh. Output model disimpan ke `/kaggle/working/best_absa_indobert_finetune2_v3_split`.


## 1. Setup

In [ ]:
import sys, subprocess
def pipi(p): subprocess.run([sys.executable,'-m','pip','install','-q',*p])
# PENTING: jangan menurunkan numpy. Kaggle berbasis numpy 2.x; transformers==4.44 (butuh numpy<2)
# akan menurunkan numpy -> error 'numpy.dtype size changed' saat import pandas.
# sentencepiece TETAP dipasang (dibutuhkan tokenizer indobert-lite). datasets/evaluate tidak dipakai.
pipi(['transformers>=4.46,<5','accelerate>=0.34','sentencepiece'])
import numpy, transformers
print('done. numpy', numpy.__version__, '| transformers', transformers.__version__)
print('Jika numpy < 2.0 di sini: menu Run -> Factory reset, lalu jalankan ulang dari sel ini.')

In [ ]:

import os, json, random, gc, itertools, time, traceback
from pathlib import Path

import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, BertTokenizer, AutoModel, get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix


def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))


## 2. Load data

In [ ]:
ASPECTS = ['Lokasi', 'Kenyamanan', 'Pelayanan', 'Kebersihan', 'Harga', 'Makanan', 'Fasilitas']
LABEL2ID = {'none': 0, 'positif': 1, 'negatif': 2, 'netral': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 4
LABEL_NAMES = ['none', 'positif', 'negatif', 'netral']

# Finetune2 rerun dengan dataset split V3 dari Kaggle: absa_santika_split_v3.
# Catatan Kaggle: nama dataset dengan underscore sering di-mount sebagai folder ber-hyphen,
# sedangkan isi foldernya bisa tetap underscore. Karena itu resolver dibuat fleksibel.
DATASET_VERSION = 'v3_final_audited'
SPLIT_VERSION = 'absa_santika_split_v3'
EXPECTED_SPLIT_ROWS = {'train': 11797, 'validation': 1475, 'test': 1475}

KAGGLE_INPUT = Path('/kaggle/input')
LOCAL_PROJECT_ROOT = Path(r'C:\Users\cencen04_\Downloads\ABSA Hotel Santika')

SPLIT_DIR_CANDIDATES = [
    # Bentuk paling mungkin dari screenshot: absa_santika_split_v3.
    KAGGLE_INPUT / 'absa-santika-split-v3' / 'absa_santika_split_v3',
    KAGGLE_INPUT / 'absa-santika-split-v3' / 'absa-santika-split-v3',
    KAGGLE_INPUT / 'absa-santika-split-v3',
    KAGGLE_INPUT / 'absa_santika_split_v3' / 'absa_santika_split_v3',
    KAGGLE_INPUT / 'absa_santika_split_v3',
    # Bentuk jika dataset ditambahkan sebagai /datasets/<owner>/<slug>/...
    KAGGLE_INPUT / 'datasets' / 'kcent-gosling' / 'absa-santika-split-v3' / 'absa_santika_split_v3',
    KAGGLE_INPUT / 'datasets' / 'kcent-gosling' / 'absa-santika-split-v3',
    KAGGLE_INPUT / 'datasets' / 'korneliusvincent' / 'absa-santika-split-v3' / 'absa_santika_split_v3',
    KAGGLE_INPUT / 'datasets' / 'korneliusvincent' / 'absa-santika-split-v3',
    # Fallback lokal untuk cek notebook di laptop.
    LOCAL_PROJECT_ROOT / 'absa_santika_split_v3',
    LOCAL_PROJECT_ROOT / 'Data Splitting' / 'absa_santika_split_v3',
]


def resolve_input_path(filename, preferred_keywords=('v3', 'split'), required=True):
    for directory in SPLIT_DIR_CANDIDATES:
        path = Path(directory) / filename
        if path.exists():
            return path

    candidates = []
    search_roots = [KAGGLE_INPUT, LOCAL_PROJECT_ROOT, Path.cwd(), Path.cwd().parent]
    for root in search_roots:
        root = Path(root)
        if root.exists():
            candidates.extend(root.rglob(filename))

    def score(path):
        low = str(path).lower()
        score_value = sum(keyword.lower() in low for keyword in preferred_keywords)
        # Hindari salah ambil split lama jika split v3 tersedia.
        if 'absa-santika-split-v3' in low or 'absa_santika_split_v3' in low:
            score_value += 10
        if 'absa-santika-split' in low and 'v3' not in low:
            score_value -= 5
        return score_value

    candidates = sorted(set(candidates), key=lambda x: (-score(x), str(x).lower()))
    if candidates:
        picked = candidates[0]
        print(f'[WARN] Menggunakan kandidat path: {picked}')
        return picked

    if required:
        raise FileNotFoundError(
            f'Tidak menemukan {filename}. Pastikan Kaggle dataset absa_santika_split_v3 sudah ditambahkan ke Notebook Input. '
            f'File yang wajib ada: train.csv, validation.csv, test.csv.'
        )
    return None


def load_csv(path):
    return pd.read_csv(path, encoding='utf-8-sig', dtype=str).fillna('')


def nlab(v):
    v = str(v).strip().lower()
    return v if v in LABEL2ID else 'none'


train_path = resolve_input_path('train.csv', preferred_keywords=('v3', 'split', 'train'), required=True)
val_path = resolve_input_path('validation.csv', preferred_keywords=('v3', 'split', 'validation'), required=True)
test_path = resolve_input_path('test.csv', preferred_keywords=('v3', 'split', 'test'), required=True)
manifest_path = resolve_input_path('split_manifest.json', preferred_keywords=('v3', 'split', 'manifest'), required=False)
no_aspect_path = resolve_input_path('no_aspect.csv', preferred_keywords=('v3', 'split', 'no_aspect'), required=False)

train_df = load_csv(train_path)
val_df = load_csv(val_path)
test_df = load_csv(test_path)
# Split v3 utama tidak membutuhkan no_aspect untuk training. Kalau file ada, tetap dipakai untuk audit false aspect.
no_aspect_df = load_csv(no_aspect_path) if no_aspect_path else pd.DataFrame(columns=train_df.columns)

split_manifest = None
if manifest_path is not None:
    with open(manifest_path, 'r', encoding='utf-8') as f:
        split_manifest = json.load(f)

print('Loaded absa_santika_split_v3 for finetune2 rerun:')
print('  train     :', train_path)
print('  validation:', val_path)
print('  test      :', test_path)
print('  manifest  :', manifest_path if manifest_path else 'not found / optional')
print('  no_aspect :', no_aspect_path if no_aspect_path else 'not found / optional')

TEXT_COL = 'Text_Review' if 'Text_Review' in train_df.columns else 'text_review'
ID_COL = 'ID_Review' if 'ID_Review' in train_df.columns else ('review_id' if 'review_id' in train_df.columns else None)
REQUIRED_COLS = [TEXT_COL] + ASPECTS


def validate_frame(df, name, require_non_empty=True):
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f'Kolom hilang di {name}: {missing}. Kolom tersedia: {list(df.columns)}')
    if len(df) == 0:
        if require_non_empty:
            raise ValueError(f'{name} kosong.')
        print(f'[WARN] {name} kosong.')
        return
    empty_text = df[TEXT_COL].astype(str).str.strip().eq('').sum()
    if empty_text:
        raise ValueError(f'{name} memiliki {empty_text} Text_Review kosong.')
    invalid = {}
    for aspect in ASPECTS:
        values = df[aspect].astype(str).str.strip().str.lower().replace({'': 'none', 'nan': 'none', '-': 'none'})
        bad = sorted(set(values) - set(LABEL2ID.keys()))
        if bad:
            invalid[aspect] = bad[:10]
        df[aspect] = values
    if invalid:
        raise ValueError(f'Label invalid di {name}: {invalid}')


validate_frame(train_df, 'train')
validate_frame(val_df, 'validation')
validate_frame(test_df, 'test')
if len(no_aspect_df) > 0:
    validate_frame(no_aspect_df, 'no_aspect', require_non_empty=False)

if ID_COL is not None:
    for frame in [train_df, val_df, test_df, no_aspect_df]:
        if ID_COL in frame.columns:
            frame[ID_COL] = frame[ID_COL].astype(str)
    train_ids, val_ids, test_ids = set(train_df[ID_COL]), set(val_df[ID_COL]), set(test_df[ID_COL])
    overlaps = {
        'train_val': len(train_ids & val_ids),
        'train_test': len(train_ids & test_ids),
        'val_test': len(val_ids & test_ids),
    }
    print('Split overlap:', overlaps)
    if any(overlaps.values()):
        raise ValueError(f'Data leakage antar split terdeteksi: {overlaps}')

actual_rows = {'train': len(train_df), 'validation': len(val_df), 'test': len(test_df)}
print('Actual rows:', actual_rows)
if actual_rows != EXPECTED_SPLIT_ROWS:
    print('[WARN] Jumlah row berbeda dari split v3 yang diharapkan:', EXPECTED_SPLIT_ROWS)

if split_manifest:
    print('Split manifest method:', split_manifest.get('method', '-'))
    print('Split manifest max diff:', split_manifest.get('max_abs_pct_diff_aspect_label', '-'))

summary_df = pd.DataFrame({
    'split': ['train', 'validation', 'test', 'no_aspect'],
    'rows': [len(train_df), len(val_df), len(test_df), len(no_aspect_df)],
})
summary_df['percentage'] = summary_df['rows'] / max(summary_df.loc[summary_df['split'] != 'no_aspect', 'rows'].sum(), 1)
display(summary_df)


In [ ]:

def targets(df):
    Y = np.zeros((len(df), len(ASPECTS)), dtype=np.int64)
    for i, aspect in enumerate(ASPECTS):
        Y[:, i] = df[aspect].map(nlab).map(LABEL2ID).astype(np.int64).values
    return Y


y_train, y_val, y_test = targets(train_df), targets(val_df), targets(test_df)
y_no_aspect = targets(no_aspect_df) if len(no_aspect_df) else np.zeros((0, len(ASPECTS)), dtype=np.int64)

# Class weighting memakai sqrt inverse-frequency + clipping agar kelas sangat kecil
# tidak memberi loss terlalu ekstrem. Ini lebih stabil daripada inverse penuh.
CLASS_WEIGHT_MODE = 'sqrt_capped'
CLASS_WEIGHT_MAX = 4.0
CLASS_WEIGHT_MIN = 0.25


def cls_weights(y, mode=CLASS_WEIGHT_MODE):
    W = np.ones((len(ASPECTS), NUM_CLASSES), dtype=np.float32)
    for i in range(len(ASPECTS)):
        vals, cnts = np.unique(y[:, i], return_counts=True)
        freq = np.zeros(NUM_CLASSES, dtype=np.float32)
        freq[vals] = cnts
        freq[freq == 0] = 1.0
        inv = freq.sum() / (NUM_CLASSES * freq)
        if mode == 'sqrt_capped':
            w = np.sqrt(inv)
            w = np.clip(w, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX)
            w = w / w.mean()
        elif mode == 'inverse_capped':
            w = np.clip(inv, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX)
            w = w / w.mean()
        else:
            w = np.ones(NUM_CLASSES, dtype=np.float32)
        W[i] = w
    return torch.tensor(W, dtype=torch.float32).to(DEVICE)


CLASS_W = cls_weights(y_train)
class_weight_df = pd.DataFrame(
    CLASS_W.detach().cpu().numpy(),
    index=ASPECTS,
    columns=LABEL_NAMES,
).round(3)
print('Class weight mode:', CLASS_WEIGHT_MODE)
display(class_weight_df)

# Ringkas distribusi label per split untuk audit cepat.
def label_distribution_df(name, y):
    rows = []
    for i, aspect in enumerate(ASPECTS):
        counts = pd.Series(y[:, i]).map(ID2LABEL).value_counts().reindex(LABEL_NAMES, fill_value=0)
        rows.append({'split': name, 'aspect': aspect, **counts.to_dict()})
    return pd.DataFrame(rows)

dist_df = pd.concat([
    label_distribution_df('train', y_train),
    label_distribution_df('validation', y_val),
    label_distribution_df('test', y_test),
], ignore_index=True)
display(dist_df)


## 3. Dataset & Model (multi-head) + freeze option

In [ ]:
class DS(Dataset):
    def __init__(self,texts,Y,tok,ml): self.t=list(texts); self.Y=Y; self.tok=tok; self.ml=ml
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(self.t[i],truncation=True,max_length=self.ml,padding='max_length',return_tensors='pt')
        it={k:v.squeeze(0) for k,v in e.items()}; it['labels']=torch.tensor(self.Y[i]); return it

class MultiHead(nn.Module):
    def __init__(self,name,na,nc,dropout=0.1,freeze=0):
        super().__init__(); self.enc=AutoModel.from_pretrained(name)
        h=self.enc.config.hidden_size; self.drop=nn.Dropout(dropout)
        self.heads=nn.ModuleList([nn.Linear(h,nc) for _ in range(na)])
        if freeze>0:
            for p in self.enc.embeddings.parameters(): p.requires_grad=False
            layers=getattr(self.enc.encoder,'layer',[])
            for l in layers[:freeze]:
                for p in l.parameters(): p.requires_grad=False
    def forward(self,input_ids,attention_mask,token_type_ids=None):
        o=self.enc(input_ids=input_ids,attention_mask=attention_mask)
        cls=self.drop(o.last_hidden_state[:,0])
        return torch.stack([hd(cls) for hd in self.heads],dim=1)

## 4. Training + Early Stopping + evaluasi

In [ ]:

def make_loss(cw, smoothing):
    if cw:
        outs = [nn.CrossEntropyLoss(weight=CLASS_W[i], label_smoothing=smoothing) for i in range(len(ASPECTS))]
    else:
        outs = [nn.CrossEntropyLoss(label_smoothing=smoothing) for _ in range(len(ASPECTS))]

    def fn(logits, labels):
        return sum(outs[i](logits[:, i, :], labels[:, i]) for i in range(len(ASPECTS))) / len(ASPECTS)
    return fn


def aspect_metrics(y_true, y_pred, aspect_idx):
    yt = y_true[:, aspect_idx]
    yp = y_pred[:, aspect_idx]
    present_mask = yt != LABEL2ID['none']
    true_none_mask = yt == LABEL2ID['none']

    macro_f1 = f1_score(yt, yp, labels=[0, 1, 2, 3], average='macro', zero_division=0)
    weighted_f1 = f1_score(yt, yp, labels=[0, 1, 2, 3], average='weighted', zero_division=0)
    acc = accuracy_score(yt, yp)
    non_none_macro_f1 = f1_score(yt, yp, labels=[1, 2, 3], average='macro', zero_division=0)
    aspect_detection_f1 = f1_score((yt != 0).astype(int), (yp != 0).astype(int), zero_division=0)
    false_aspect_rate = float((yp[true_none_mask] != 0).mean()) if true_none_mask.any() else 0.0
    sentiment_macro_f1_present = (
        f1_score(yt[present_mask], yp[present_mask], labels=[1, 2, 3], average='macro', zero_division=0)
        if present_mask.any() else 0.0
    )
    return {
        'macro_f1': float(macro_f1),
        'weighted_f1': float(weighted_f1),
        'acc': float(acc),
        'non_none_macro_f1': float(non_none_macro_f1),
        'aspect_detection_f1': float(aspect_detection_f1),
        'false_aspect_rate': float(false_aspect_rate),
        'sentiment_macro_f1_present': float(sentiment_macro_f1_present),
    }


@torch.no_grad()
def evaluate(model, loader, ret=False):
    model.eval()
    logits_all, labels_all = [], []
    for batch in loader:
        labels = batch.pop('labels').to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'token_type_ids']}
        logits_all.append(model(**inputs).cpu())
        labels_all.append(labels.cpu())

    if not logits_all:
        empty = {'macro_f1': 0.0, 'weighted_f1': 0.0, 'acc': 0.0, 'non_none_macro_f1': 0.0,
                 'aspect_detection_f1': 0.0, 'false_aspect_rate': 0.0,
                 'sentiment_macro_f1_present': 0.0, 'per_aspect': {}}
        return (empty, np.empty((0, len(ASPECTS))), np.empty((0, len(ASPECTS)))) if ret else empty

    L = torch.cat(logits_all)
    Y = torch.cat(labels_all).numpy()
    P = L.argmax(-1).numpy()

    per = {aspect: aspect_metrics(Y, P, i) for i, aspect in enumerate(ASPECTS)}
    r = {
        'macro_f1': float(np.mean([per[a]['macro_f1'] for a in ASPECTS])),
        'weighted_f1': float(np.mean([per[a]['weighted_f1'] for a in ASPECTS])),
        'acc': float(np.mean([per[a]['acc'] for a in ASPECTS])),
        'non_none_macro_f1': float(np.mean([per[a]['non_none_macro_f1'] for a in ASPECTS])),
        'aspect_detection_f1': float(np.mean([per[a]['aspect_detection_f1'] for a in ASPECTS])),
        'false_aspect_rate': float(np.mean([per[a]['false_aspect_rate'] for a in ASPECTS])),
        'sentiment_macro_f1_present': float(np.mean([per[a]['sentiment_macro_f1_present'] for a in ASPECTS])),
        'per_aspect': per,
    }
    return (r, P, Y) if ret else r


In [ ]:
def build_opt(model,cfg):
    if cfg.get('optimizer','adamw')=='adafactor':
        from transformers.optimization import Adafactor
        return Adafactor(model.parameters(),lr=cfg['lr'],scale_parameter=False,relative_step=False,warmup_init=False)
    return AdamW(model.parameters(),lr=cfg['lr'],weight_decay=cfg.get('weight_decay',0.01))

def build_sched(opt,cfg,total):
    w=int(cfg.get('warmup_ratio',0.1)*total)
    if cfg.get('scheduler','linear')=='cosine':
        return get_cosine_schedule_with_warmup(opt,w,total)
    return get_linear_schedule_with_warmup(opt,w,total)

def train_cfg(cfg, verbose=True):
    set_seed(cfg.get('seed',42))
    # indobert-lite = arsitektur ALBERT tapi vocab WordPiece (vocab.txt),
    # jadi AutoTokenizer keliru memilih AlbertTokenizer (cari SentencePiece) -> 'not a string'.
    # Pakai BertTokenizer untuk model lite, AutoTokenizer untuk lainnya.
    name=cfg['model_name']
    tok=(BertTokenizer if 'lite' in name.lower() else AutoTokenizer).from_pretrained(name)
    trL=DataLoader(DS(train_df[TEXT_COL].tolist(),y_train,tok,cfg['max_len']),batch_size=cfg['batch_size'],shuffle=True)
    vaL=DataLoader(DS(val_df[TEXT_COL].tolist(),y_val,tok,cfg['max_len']),batch_size=cfg['batch_size'])
    model=MultiHead(cfg['model_name'],len(ASPECTS),NUM_CLASSES,cfg.get('dropout',0.1),cfg.get('freeze_layers',0)).to(DEVICE)
    loss_fn=make_loss(cfg.get('class_weight',False),cfg.get('label_smoothing',0.0))
    opt=build_opt(model,cfg); total=len(trL)*cfg['max_epochs']; sched=build_sched(opt,cfg,total)
    best=-1; best_state=None; bad=0; patience=cfg.get('patience',2); hist=[]
    for ep in range(cfg['max_epochs']):
        model.train()
        for b in trL:
            lb=b.pop('labels').to(DEVICE); b={k:v.to(DEVICE) for k,v in b.items() if k in ['input_ids','attention_mask','token_type_ids']}
            loss=loss_fn(model(**b),lb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); sched.step(); opt.zero_grad()
        r=evaluate(model,vaL); hist.append(r['macro_f1'])
        if verbose: print(f"   ep{ep+1}/{cfg['max_epochs']} valF1={r['macro_f1']:.4f}")
        if r['macro_f1']>best+1e-4:
            best=r['macro_f1']; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience:
                if verbose: print(f'   early stop @ epoch {ep+1}')
                break
    if best_state: model.load_state_dict(best_state)
    return model,tok,best,hist,(ep+1)

## 5. Ruang pencarian (Grid & Random)

Atur `SEARCH_MODE`:
- `'curated'` : daftar konfigurasi pilihan (cepat, direkomendasikan untuk mulai).
- `'random'`  : sampel acak dari ruang besar sebanyak `N_RANDOM` (Bergstra & Bengio 2012).
- `'grid'`    : full cartesian dari `GRID` (HATI-HATI: bisa sangat banyak & lama).

In [ ]:
P1='indobenchmark/indobert-base-p1'
LITE='indobenchmark/indobert-lite-base-p1'
LEM='indolem/indobert-base-uncased'

# ---- ruang nilai untuk random/grid ----
SPACE = dict(
    model_name=[P1, LEM, LITE],
    lr=[1e-5,2e-5,3e-5,5e-5],
    batch_size=[8,16,32],
    max_epochs=[4,6,8,10],
    max_len=[96,128,160],
    warmup_ratio=[0.0,0.06,0.1],
    weight_decay=[0.0,0.01,0.1],
    dropout=[0.1,0.2,0.3],
    scheduler=['linear','cosine'],
    optimizer=['adamw','adafactor'],
    label_smoothing=[0.0,0.1],
    freeze_layers=[0,6],
    class_weight=[False,True],
    seed=[42,7,123],
)

# ---- daftar curated (cepat, sudah mencakup variasi utama) ----
CURATED=[
    dict(name='c1_p1_base',        model_name=P1, lr=2e-5, batch_size=16, max_epochs=8,  max_len=128, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.1, scheduler='linear', optimizer='adamw', label_smoothing=0.0, freeze_layers=0, class_weight=False, patience=2),
    dict(name='c2_p1_lr3',         model_name=P1, lr=3e-5, batch_size=16, max_epochs=8,  max_len=128, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.1, scheduler='linear', optimizer='adamw', label_smoothing=0.0, freeze_layers=0, class_weight=False, patience=2),
    dict(name='c3_p1_bs32_cos',    model_name=P1, lr=3e-5, batch_size=32, max_epochs=10, max_len=128, warmup_ratio=0.06, weight_decay=0.01, dropout=0.1, scheduler='cosine', optimizer='adamw', label_smoothing=0.0, freeze_layers=0, class_weight=False, patience=3),
    dict(name='c4_p1_cw_ls',       model_name=P1, lr=3e-5, batch_size=16, max_epochs=10, max_len=128, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.2, scheduler='linear', optimizer='adamw', label_smoothing=0.1, freeze_layers=0, class_weight=True,  patience=3),
    dict(name='c5_p1_bs8_lr1',     model_name=P1, lr=1e-5, batch_size=8,  max_epochs=8,  max_len=160, warmup_ratio=0.1,  weight_decay=0.1,  dropout=0.1, scheduler='cosine', optimizer='adamw', label_smoothing=0.0, freeze_layers=0, class_weight=False, patience=2),
    dict(name='c6_p1_freeze6',     model_name=P1, lr=5e-5, batch_size=32, max_epochs=10, max_len=128, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.1, scheduler='linear', optimizer='adamw', label_smoothing=0.0, freeze_layers=6, class_weight=False, patience=3),
    dict(name='c7_lem_base',       model_name=LEM, lr=3e-5, batch_size=16, max_epochs=8, max_len=128, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.1, scheduler='linear', optimizer='adamw', label_smoothing=0.0, freeze_layers=0, class_weight=False, patience=2),
    dict(name='c8_lem_cw',         model_name=LEM, lr=3e-5, batch_size=16, max_epochs=10, max_len=128, warmup_ratio=0.1, weight_decay=0.01, dropout=0.2, scheduler='cosine', optimizer='adamw', label_smoothing=0.1, freeze_layers=0, class_weight=True, patience=3),
    dict(name='c9_lite_fast',      model_name=LITE, lr=3e-5, batch_size=32, max_epochs=10, max_len=128, warmup_ratio=0.1, weight_decay=0.01, dropout=0.1, scheduler='linear', optimizer='adamw', label_smoothing=0.0, freeze_layers=0, class_weight=False, patience=3),
    dict(name='c10_p1_adafactor',  model_name=P1, lr=3e-5, batch_size=16, max_epochs=8,  max_len=128, warmup_ratio=0.1,  weight_decay=0.0,  dropout=0.1, scheduler='linear', optimizer='adafactor', label_smoothing=0.0, freeze_layers=0, class_weight=False, patience=2),
]

In [ ]:
# Mode final: hanya jalankan satu konfigurasi terpilih untuk model akhir skripsi.
SEARCH_MODE = 'final_only'
FINAL_MODEL_NAME = 'c4_p1_cw_ls'
N_RANDOM = 12

def sample_random(n, seed=42):
    rng=random.Random(seed); out=[]
    for j in range(n):
        cfg={k:rng.choice(v) for k,v in SPACE.items()}
        cfg['patience']=2; cfg['name']=f'rand_{j+1}'
        out.append(cfg)
    return out

def build_grid(keys):
    base=dict(model_name=P1,max_len=128,warmup_ratio=0.1,weight_decay=0.01,dropout=0.1,
              scheduler='linear',optimizer='adamw',label_smoothing=0.0,freeze_layers=0,
              class_weight=False,seed=42,patience=2)
    vals=[SPACE[k] for k in keys]; out=[]
    for combo in itertools.product(*vals):
        c=dict(base); c.update(dict(zip(keys,combo)))
        c['name']='grid_'+'_'.join(f'{k}{c[k]}' for k in keys)
        out.append(c)
    return out

if SEARCH_MODE == 'final_only':
    EXPERIMENTS = [cfg for cfg in CURATED if cfg['name'] == FINAL_MODEL_NAME]
elif SEARCH_MODE == 'curated':
    EXPERIMENTS = CURATED
elif SEARCH_MODE == 'random':
    EXPERIMENTS = sample_random(N_RANDOM)
else:
    EXPERIMENTS = build_grid(['lr','batch_size','max_epochs'])

if len(EXPERIMENTS) != 1:
    raise ValueError(f'Konfigurasi final {FINAL_MODEL_NAME} tidak ditemukan atau tidak unik.')

print('Mode:', SEARCH_MODE)
print('Model final:', EXPERIMENTS[0]['name'])
print('Total eksperimen:', len(EXPERIMENTS))
EXPERIMENTS[0]


## 6. Jalankan eksperimen

In [ ]:

def is_oom_error(err):
    msg = str(err).lower()
    return any(token in msg for token in [
        'out of memory',
        'cuda error: out of memory',
        'cublas_status_alloc_failed',
        'cuda out of memory',
    ])


results = []
best = None
t0 = time.time()
for cfg in EXPERIMENTS:
    print('=' * 64)
    print('RUN:', cfg['name'])
    try:
        model, tok, vf1, hist, ran_ep = train_cfg(cfg)
    except RuntimeError as e:
        if is_oom_error(e):
            print('  SKIP OOM:', str(e)[:180])
            torch.cuda.empty_cache(); gc.collect()
            continue
        print(traceback.format_exc())
        raise

    row = {k: cfg.get(k) for k in [
        'name', 'model_name', 'lr', 'batch_size', 'max_epochs', 'max_len',
        'warmup_ratio', 'weight_decay', 'dropout', 'scheduler', 'optimizer',
        'label_smoothing', 'freeze_layers', 'class_weight', 'seed'
    ]}
    row['epochs_ran'] = ran_ep
    row['val_macro_f1'] = vf1
    results.append(row)
    print(f'  >> val_macroF1={vf1:.4f} (ran {ran_ep} epoch)')

    if best is None or vf1 > best['vf1']:
        if best is not None:
            del best['model']
            gc.collect(); torch.cuda.empty_cache()
        best = {'cfg': cfg, 'vf1': vf1, 'model': model, 'tok': tok}
    else:
        del model
        gc.collect(); torch.cuda.empty_cache()

print(f'Total waktu: {(time.time() - t0) / 60:.1f} menit')
if not results or best is None:
    raise RuntimeError('Tidak ada eksperimen yang berhasil. Cek path data, koneksi model Hugging Face, atau konfigurasi GPU.')
res_df = pd.DataFrame(results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
res_df


## 7. Evaluasi model terbaik di TEST

In [ ]:

if best is None:
    raise RuntimeError('Belum ada model terbaik. Jalankan cell eksperimen terlebih dahulu.')

bc = best['cfg']
bm = best['model']
bt = best['tok']
print('Terbaik:', bc['name'], '| val_macroF1=', round(best['vf1'], 4))

teL = DataLoader(DS(test_df[TEXT_COL].tolist(), y_test, bt, bc['max_len']), batch_size=bc['batch_size'])
tr, P, Y = evaluate(bm, teL, ret=True)
print(
    'TEST | macroF1=', round(tr['macro_f1'], 4),
    '| weightedF1=', round(tr['weighted_f1'], 4),
    '| nonNoneF1=', round(tr['non_none_macro_f1'], 4),
    '| aspectDetectF1=', round(tr['aspect_detection_f1'], 4),
    '| falseAspectRate=', round(tr['false_aspect_rate'], 4),
    '| acc=', round(tr['acc'], 4),
)
for aspect, value in tr['per_aspect'].items():
    print(
        f"  {aspect:12s} macroF1={value['macro_f1']:.4f} "
        f"nonNoneF1={value['non_none_macro_f1']:.4f} "
        f"detectF1={value['aspect_detection_f1']:.4f} "
        f"falseAspect={value['false_aspect_rate']:.4f} "
        f"acc={value['acc']:.4f}"
    )

# Audit no-aspect: mengukur kecenderungan model mengarang aspek pada review umum.
no_aspect_summary = None
if len(no_aspect_df) > 0:
    noL = DataLoader(DS(no_aspect_df[TEXT_COL].tolist(), y_no_aspect, bt, bc['max_len']), batch_size=bc['batch_size'])
    no_tr, noP, noY = evaluate(bm, noL, ret=True)
    no_aspect_summary = {
        'n_rows': int(len(no_aspect_df)),
        'false_aspect_rate_mean': float(no_tr['false_aspect_rate']),
        'per_aspect_false_aspect_rate': {
            aspect: float(no_tr['per_aspect'][aspect]['false_aspect_rate']) for aspect in ASPECTS
        },
    }
    print('\nNO-ASPECT AUDIT')
    print('Rows:', no_aspect_summary['n_rows'])
    print('Mean false aspect rate:', round(no_aspect_summary['false_aspect_rate_mean'], 4))
    for aspect, rate in no_aspect_summary['per_aspect_false_aspect_rate'].items():
        print(f'  {aspect:12s}: {rate:.4f}')
else:
    print('\nNO-ASPECT AUDIT skipped: no_aspect.csv tidak ditemukan / kosong.')


In [ ]:

test_reports = {}
test_confusion_matrices = {}
for i, aspect in enumerate(ASPECTS):
    print('=' * 48)
    print(aspect)
    print(classification_report(
        Y[:, i], P[:, i],
        labels=[0, 1, 2, 3],
        target_names=LABEL_NAMES,
        zero_division=0,
    ))
    test_reports[aspect] = classification_report(
        Y[:, i], P[:, i],
        labels=[0, 1, 2, 3],
        target_names=LABEL_NAMES,
        zero_division=0,
        output_dict=True,
    )
    cm = confusion_matrix(Y[:, i], P[:, i], labels=[0, 1, 2, 3])
    test_confusion_matrices[aspect] = cm.tolist()
    display(pd.DataFrame(cm, index=[f'true_{x}' for x in LABEL_NAMES], columns=[f'pred_{x}' for x in LABEL_NAMES]))


## 8. Visualisasi perbandingan eksperimen

In [ ]:

import matplotlib.pyplot as plt

if len(res_df) == 0:
    print('Tidak ada hasil eksperimen untuk divisualisasikan.')
else:
    top = res_df.head(15)[::-1]
    plt.figure(figsize=(9, 6))
    plt.barh(top['name'], top['val_macro_f1'])
    plt.xlabel('Validation macro-F1')
    plt.title('Perbandingan Eksperimen (Top 15)')
    plt.tight_layout()
    plt.show()


## 9. Simpan model & hasil

In [ ]:
# Folder output ini yang harus muncul di tab Output Kaggle setelah Save Version berhasil.
OUT = '/kaggle/working/best_absa_indobert_finetune2_v3_split'
os.makedirs(OUT, exist_ok=True)

torch.save(bm.state_dict(), os.path.join(OUT, 'model_state.pt'))
bt.save_pretrained(OUT)

payload = {
    'model_name': bc['model_name'],
    'dataset_version': DATASET_VERSION,
    'split_version': SPLIT_VERSION,
    'split_manifest': split_manifest,
    'train_rows': int(len(train_df)),
    'validation_rows': int(len(val_df)),
    'test_rows': int(len(test_df)),
    'train_path': str(train_path),
    'validation_path': str(val_path),
    'test_path': str(test_path),
    'manifest_path': str(manifest_path) if manifest_path else None,
    'aspects': ASPECTS,
    'label2id': LABEL2ID,
    'id2label': ID2LABEL,
    'max_len': bc['max_len'],
    'best_cfg': dict(bc),
    'class_weight_mode': CLASS_WEIGHT_MODE,
    'class_weight_max': CLASS_WEIGHT_MAX,
    'class_weight_min': CLASS_WEIGHT_MIN,
    'val_macro_f1': float(best['vf1']),
    'test': tr,
    'no_aspect_audit': no_aspect_summary,
}
with open(os.path.join(OUT, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

res_df.to_csv(os.path.join(OUT, 'experiment_results.csv'), index=False)
if 'test_reports' in globals():
    with open(os.path.join(OUT, 'test_classification_reports.json'), 'w', encoding='utf-8') as f:
        json.dump(test_reports, f, indent=2, ensure_ascii=False)
if 'test_confusion_matrices' in globals():
    with open(os.path.join(OUT, 'test_confusion_matrices.json'), 'w', encoding='utf-8') as f:
        json.dump(test_confusion_matrices, f, indent=2, ensure_ascii=False)
if len(no_aspect_df) > 0 and 'noP' in globals():
    no_aspect_pred = no_aspect_df.copy()
    for i, aspect in enumerate(ASPECTS):
        no_aspect_pred[f'pred_{aspect}'] = [ID2LABEL[int(x)] for x in noP[:, i]]
    no_aspect_pred.to_csv(os.path.join(OUT, 'no_aspect_predictions.csv'), index=False)

print('Saved ->', OUT)
print(os.listdir(OUT))


## Catatan akhir
- **Mulai dengan `SEARCH_MODE='curated'`** untuk target cepat di Kaggle T4.
- Split train/validation/test sekarang dibaca dari path Kaggle eksplisit (`absa-santika-split`) agar eksperimen reproducible dan tidak salah mengambil file bernama sama.
- `no_aspect.csv` tidak dipakai sebagai data training utama; file ini dipakai sebagai **audit hallucination / false aspect rate**, yaitu apakah model memprediksi aspek pada review yang seharusnya tidak memiliki aspek.
- Class weight memakai **sqrt inverse-frequency + clipping** agar imbalance tetap tertangani tanpa membuat kelas minoritas yang sangat kecil menjadi terlalu dominan.
- Evaluasi akhir mencakup macro-F1, weighted-F1, non-None macro-F1, aspect-detection F1, false-aspect rate, classification report, dan confusion matrix per aspek.
- Simpan hasil terbaik di `/kaggle/working/best_absa_indobert`, lalu download folder output tersebut untuk dokumentasi eksperimen skripsi.
